# RQ2: Search Strategy Evaluation

This notebook analyzes the impact of different search strategies on the Branch and Bound solver performance. We compare four search strategies:

- **DFS (Depth-First Search)**: Explores children ordered by bound quality
- **BFS (Breadth-First Search)**: Explores nodes level by level, cheapest first
- **DFS+BFS**: Hybrid approach combining both strategies
- **Random**: Randomly selects the next node to explore

We evaluate these strategies across multiple metrics including runtime, solution quality (upper/lower bounds), and the number of instances solved to optimality.

## Setup and Imports

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from algbench import read_as_pandas
from tspn_bnb2.misc.paper_style import FULLWIDEFIGURE, HALFWIDEFIGURE, init_params
from tspn_bnb2.schemas import AnnotatedInstance, AnnotatedSolution

init_params()

In [ ]:
search_strategy_labels = {
    "CheapestChildDepthFirst": "DFS",
    "CheapestBreadthFirst": "BFS",
    "DfsBfs": "DFS+BFS",
    "Random": "Random",
}

## Data Loading

Load benchmark results from the `results_search_strategy` directory. Each row contains:
- The solved instance and its solution
- Upper and lower bounds achieved
- Whether optimality was proven
- The search strategy used and optimality tolerance (eps)

In [ ]:
def read_row(row):
    instance = AnnotatedInstance.model_validate_json(
        row["parameters"]["args"]["kwargs"]["instance_json"]
    )
    solution = (
        AnnotatedSolution.model_validate_json(row["result"]["solution"])
        if row["result"]["solution"]
        else None
    )

    if solution is None:
        print(
            row["result"]["error"],
            row["parameters"]["args"]["kwargs"]["instance_name"],
            row["parameters"]["args"]["alg_params"],
        )
        return None

    return {
        "solution": solution,
        "upper_bound": solution.upper_bound,
        "lower_bound": solution.lower_bound,
        "gap": (solution.upper_bound - solution.lower_bound) / solution.upper_bound,
        "is_optimal": solution.is_optimal,
        "instance_name": row["parameters"]["args"]["kwargs"]["instance_name"],
        "instance": instance,
        "solve_time": row["result"].get("solve_time"),
        "n": instance.num_polygons(),
        "search_strategy": search_strategy_labels[
            row["parameters"]["args"]["alg_params"]["search"]
        ],
        "eps": row["parameters"]["args"]["alg_params"]["eps"],
    }


benchmark = read_as_pandas("results_search_strategy", read_row)
print(benchmark.columns)
benchmark = benchmark.sort_values(by=["search_strategy"], ascending=False)

print("loaded", len(benchmark), "runs")

## Optimality Summary

Count how many instances each search strategy solved to optimality.

In [ ]:
benchmark.groupby(by=["search_strategy"])["is_optimal"].value_counts()

## Solution Validation

Verify that all solutions are feasible and consistent across strategies. For each instance, we check:
1. The solution trajectory is feasible (visits all neighborhoods)
2. Solutions from different strategies are plausibly consistent (bounds agree within tolerance)

In [ ]:
# validate that solutions are correct.
for eps in benchmark["eps"].unique():
    n_checks = 0
    current_bench = benchmark[benchmark["eps"] == eps]
    for _, row in current_bench.iterrows():
        solution = row["solution"]
        if solution is None or solution.trajectory.is_empty:
            continue
        assert solution.check_feasibility(row["instance"], eps=1), row["search_strategy"]
        same_instance = current_bench[current_bench["instance_name"] == row["instance_name"]]
        for _, other in same_instance.iterrows():
            if other["solution"] is None:
                continue

            check = solution.plausibility_check(
                other["solution"],
                eps=eps,
            )
            assert check["valid"], f"Check failed for {row['instance_name']} {check}"
            n_checks += 1

    print(n_checks, "checks succeeded")

## Runtime Analysis

Compare solve times across search strategies. The boxplots show runtime distribution, with the percentage of instances solved to optimality shown below each strategy label.

In [ ]:
fig, ax = plt.subplots(ncols=1, figsize=HALFWIDEFIGURE)

sns.boxplot(
    benchmark, x="search_strategy", y="solve_time", hue="eps", ax=ax, palette=sns.color_palette()
)

xticks = ax.get_xticks()
xticklabels = []

for label in ax.get_xticklabels():
    optimal_percentage_for_eps = {}
    for eps in sorted(benchmark["eps"].unique()):
        current_bench = benchmark[benchmark["eps"] == eps]
        solutions_for_label = current_bench[(current_bench["search_strategy"] == label.get_text())]
        optimal_percentage_for_eps[eps] = len(
            solutions_for_label[solutions_for_label["is_optimal"]]
        ) / len(solutions_for_label)

    percentages = ", ".join(
        f"{round(optimal_percentage_for_eps[eps] * 100, 1)}\\%"
        for eps in sorted(benchmark["eps"].unique())
    )
    label = label.get_text() + f"\n\\scriptsize{{[{percentages}]}}"
    xticklabels.append(label)

ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels)
ax.set_title("lower is better")
ax.set_xlabel("")
ax.set_ylabel("runtime [s]")

ax.legend(loc="upper right").set_title("tolerance")

fig.savefig("../plots/rq2_search_strategy/runtimes_combined.pdf", bbox_inches="tight")

## Performance Profiles

Dolan-Moré style performance profiles compare strategies by measuring what fraction of instances each strategy solves within a given factor of the best. A strategy that reaches 1.0 (100%) at x=1 is always the best performer.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.axes import Axes

# adapted from OR-Tools Primer: https://d-krupke.github.io/cpsat-primer/08_benchmarking.html#performance-plots-for-solution-quality-within-a-time-limit


def plot_relative_performance_profile(
    data: pd.DataFrame,
    ax: Axes,
    instance_column: str,
    strategy_column: str,
    metric_column: str,
    direction: str,
    eps: float = 0.01,
) -> Axes:
    """
    Plot a relative performance profile (Dolan–Moré style) with epsilon tolerance.

    Args:
        data: DataFrame with columns [instance, strategy, metric].
        instance_column: column identifying each problem instance.
        strategy_column: column identifying each solver/strategy.
        metric_column: column of the performance metric.
        direction: "min" if lower metric is better, "max" if higher is better.
        title: Optional plot title.
        ax: matplotlib Axes to draw into.
        eps: tolerance for comparison (e.g., 0.05 for 5%).
    """
    if direction not in ("min", "max"):
        raise ValueError("`direction` must be 'min' or 'max'.")

    # 1) Compute best value per instance
    best_val = data.groupby(instance_column)[metric_column].agg(direction)

    # 2) Pivot to get per-instance median per-strategy
    pivot = (
        data.groupby([instance_column, strategy_column])[metric_column]
        .median()
        .unstack(fill_value=np.nan)
    )

    # 3) Build relative ratio matrix
    comp = pd.DataFrame(index=pivot.index, columns=pivot.columns, dtype=float)

    # Apply epsilon tolerance correctly
    if direction == "min":
        best_val_eps = best_val * (1 + eps)
    else:  # max
        best_val_eps = best_val * (1 - eps)

    for strat in pivot.columns:
        if direction == "min":
            comp[strat] = pivot[strat] / best_val_eps
        else:  # max
            comp[strat] = best_val_eps / pivot[strat]

    comp = comp.replace([np.inf, -np.inf, 0.0], np.nan)

    # 4) Collect all distinct x-values, including baseline
    all_vals = comp.to_numpy().flatten()
    finite_vals = all_vals[np.isfinite(all_vals)]
    baseline = 1.0
    all_x = np.unique(np.sort(finite_vals))
    all_x = np.insert(all_x, 0, baseline)
    all_x = np.unique(np.sort(all_x))

    # 5) Build performance-profile DataFrame
    n_instances = comp.shape[0]
    profile = pd.DataFrame(index=all_x, columns=comp.columns, dtype=float)

    for x in all_x:
        leq = (comp <= x).sum(axis=0)
        profile.loc[x] = leq / n_instances

    # 8) Plot each solver's curve
    for strat in profile.columns:
        y = profile[strat].astype(float)
        ax.step(
            all_x,
            y,
            where="post",
            label=strat,
            linewidth=1.5,
            alpha=1.0,
        )

    ax.set_xscale("linear")
    ax.set_xlim(1 + eps, ax.get_xlim()[1])
    xlabel = "Within this factor of the best (linear scale)"

    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"\#problems [\%]")
    ax.axvline(x=baseline, color="gray", linestyle="--", alpha=0.7)
    # ax.grid(True, which="both", linestyle=":", linewidth=0.5)
    ax.legend(loc="lower right", frameon=False)

    return ax

In [ ]:
fig, ax = plt.subplots(figsize=HALFWIDEFIGURE)
eps = 0.01
plot_relative_performance_profile(
    benchmark[(benchmark["eps"] == eps)],
    ax=ax,
    instance_column="instance_name",
    direction="max",
    metric_column="lower_bound",
    strategy_column="search_strategy",
    eps=eps,
)

ax.set_title(f"performance of lower bound (opt gap {eps})")

# ax.set_xlim(1.01, ax.get_xlim()[1])

fig.savefig("../plots/rq2_search_strategy/performance_lb.pdf", bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=HALFWIDEFIGURE)

eps = 0.01
plot_relative_performance_profile(
    benchmark[(benchmark["eps"] == eps)],
    ax=ax,
    instance_column="instance_name",
    direction="min",
    metric_column="upper_bound",
    strategy_column="search_strategy",
    eps=eps,
)

ax.set_title(f"performance of upper bound (opt gap {eps})")
# ax.set_xlim(1.01, ax.get_xlim()[1])

fig.savefig("../plots/rq2_search_strategy/performance_ub.pdf", bbox_inches="tight")

## Cactus Plots

Cactus plots show how many instances are solved to optimality as a function of runtime. Curves higher and to the left indicate better performance (more instances solved faster). We compare performance at two optimality tolerances: 1% and 5%.

In [ ]:
def cactus_dataframe(
    data: pd.DataFrame,
    runtime_column: str,
    strategy_column: str,
    instance_column: str,
    flat_line_to: float | None = None,
) -> pd.DataFrame:
    """
    Convert benchmark results into a dataframe suitable for cactus plots.

    Returns a tidy dataframe with columns:
        strategy, runtime, solved
    which can be plotted with seaborn.lineplot(drawstyle="steps-post").
    """

    rows = []

    for strategy, group in data.groupby(strategy_column):
        runtimes = (
            group.groupby(instance_column)[runtime_column].min().dropna().sort_values().to_numpy()
        )

        if len(runtimes) == 0:
            continue

        solved = np.arange(1, len(runtimes) + 1)

        x = runtimes
        y = solved

        if flat_line_to is not None and runtimes[-1] < flat_line_to:
            x = np.append(x, flat_line_to)
            y = np.append(y, solved[-1])

        rows.extend(
            {
                strategy_column: strategy,
                "runtime": xi,
                "solved": yi,
            }
            for xi, yi in zip(x, y)
        )

    return pd.DataFrame(rows)

In [ ]:
from matplotlib.ticker import PercentFormatter

fig, ax = plt.subplots(figsize=FULLWIDEFIGURE)

eps1 = 0.01
eps2 = 0.05

instance_count = len(set(benchmark["instance_name"].unique()))

df1 = cactus_dataframe(
    benchmark[(benchmark["eps"] == eps1) & (benchmark["is_optimal"])],
    instance_column="instance_name",
    strategy_column="search_strategy",
    runtime_column="solve_time",
    flat_line_to=benchmark["solve_time"].max(),
)


df2 = cactus_dataframe(
    benchmark[(benchmark["eps"] == eps2) & (benchmark["is_optimal"])],
    instance_column="instance_name",
    strategy_column="search_strategy",
    runtime_column="solve_time",
    flat_line_to=benchmark["solve_time"].max(),
)

df1["optimality tolerance"] = eps1
df2["optimality tolerance"] = eps2

df = pd.concat([df1, df2])

df = df.sort_values(by=["search_strategy"], ascending=False)


sns.lineplot(
    data=df,
    x="runtime",
    y="solved",
    hue="search_strategy",
    style="optimality tolerance",
    drawstyle="steps-post",
)

# ax.set_xlim(1.01, ax.get_xlim()[1])
# ax.axhline(y=instance_count, color="black", linestyle="-.", alpha=0.5)

labels, handles = ax.get_legend_handles_labels()
fig.legend(labels[1:], handles[1:], loc="outside right center", ncol=1)
ax.legend().remove()

yticks = np.linspace(0, instance_count, 21)

ax.set_ylabel(r"solved instances [\%]")

ax.yaxis.set_major_formatter(PercentFormatter(instance_count))
ax.set_yticks(yticks)
ax.set_ylim(instance_count * 0.7, instance_count * 1.05)

# ax.set_ylim(400, instance_count + 2)
ax.set_xlim(0, 60)
ax.set_title("higher is better")
fig.subplots_adjust(right=0.74)
fig.savefig("../plots/rq2_search_strategy/cactus.pdf", bbox_inches="tight")

In [ ]:
from matplotlib.ticker import PercentFormatter
from tspn_bnb2.misc.plotting import cactus_plot

fig, axs = plt.subplots(ncols=2, figsize=FULLWIDEFIGURE)

colors = {v: f"C{i}" for i, v in enumerate(benchmark["search_strategy"].unique())}

benchmark_solved = benchmark
epsilons = [0.01, 0.05]

for i, (ax, eps) in enumerate(zip(axs, epsilons)):
    solved = benchmark_solved[benchmark_solved["eps"] == eps]

    max_gap_for_type = solved["gap"].max()
    cactus_plot(
        solved,
        ax=ax,
        instance_column="instance_name",
        strategy_column="search_strategy",
        runtime_column="gap",
        flat_line_to=max_gap_for_type,
        colors=colors,
        linestyle="-",
    )

    if i != 0:
        ax.legend().remove()
        ax.set_ylabel("")

    ax.set_title(rf"${eps * 100}\%$ optimality gap")
    ymax = solved["instance_name"].nunique()
    ax.set_ylim(ymax * 0.88, ymax * 1.01)
    ax.set_xlim(eps, max_gap_for_type)
    ax.set_xlabel(r"gap [\%]")
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))


axs[0].legend()

fig.suptitle("higher is better")
# fig.subplots_adjust(top=0.8, wspace=0.3)
fig.savefig("../plots/rq2_search_strategy/cactus_gaps_by_eps.pdf", facecolor="white")

In [ ]:
# Success Probabilities
benchmark.groupby(["search_strategy", "eps"])["is_optimal"].mean()